#Main.py


import time
import asyncio
import sys
import logging
from agent.resource_agent import ResourceAgent

# 🛠️ FIX PARA WINDOWS: Evita el RuntimeError de "Event loop is closed" al salir con CTRL+C
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

async def main():
    print("🚀 --- INICIO DEL SISTEMA MULTI-AGENTE (STORAGE) ---")

    # ---------------------------------------------------------
    # DATOS DE CONEXIÓN XMPP
    # ---------------------------------------------------------
    AGENT_JID = "agent@localhost"
    AGENT_PASS = "005060321"

    resource_agent = ResourceAgent(AGENT_JID, AGENT_PASS)

    try:
        print(f"📡 Conectando al servidor XMPP como {AGENT_JID}...")
        
        # 🛡️ auto_register=True crea la cuenta automáticamente si no existe en el servidor
        await resource_agent.start(auto_register=True)
        
        # Damos un momento para que se ejecute el setup() del agente y conecte OPC UA
        await asyncio.sleep(2)

        if resource_agent.is_alive():
            print("\n✅ --- SISTEMA ONLINE ---")
            print("Presiona CTRL+C para salir.")
            
            # Bucle infinito optimizado
            while resource_agent.is_alive():
                await asyncio.sleep(1)
                
            print("⚠️ El agente ha muerto inesperadamente.")
        else:
            print("❌ El agente no pudo arrancar (Revisa servidor XMPP local).")

    except asyncio.CancelledError:
        print("\n🛑 Tarea cancelada.")
    except KeyboardInterrupt:
        print("\n🛑 Interrupción manual detectada.")
    except Exception as e:
        print(f"\n❌ Error Crítico en main: {e}")
    finally:
        if resource_agent and resource_agent.is_alive():
            print("📉 Deteniendo agente y desconectando OPC UA...")
            await resource_agent.stop()
        print("👋 Fin del programa.")

if __name__ == "__main__":
    # Oculta los logs molestos de XMPP y SPADE (solo muestra WARNINGS para arriba)
    logging.basicConfig(level=logging.WARNING)
    
    try:
        asyncio.run(main())
    except KeyboardInterrupt:
        # Silencia el error de consola cuando el usuario sale forzosamente
        pass
```



# Arquitectura agente


================================================================================
          ARQUITECTURA DEL AGENTE INTELIGENTE - ESTACIÓN DE STORAGE
================================================================================

1. RESUMEN DEL SISTEMA
----------------------
El sistema consiste en un Agente de Recurso (Resource Agent) desarrollado sobre el
framework SPADE (Smart Python Agent Development Environment). Su objetivo es el
control, monitoreo, diagnóstico y recuperación autónoma de una estación de
almacenamiento industrial, utilizando una arquitectura de estados finitos y
sensores virtuales para la detección de fallas.

2. COMPONENTES DEL SOFTWARE (ESTRUCTURA DE ARCHIVOS)
----------------------------------------------------
- states.py:
    Define la enumeración 'AgentState' y la clase 'StateManager'.
    Implementa un mecanismo de transiciones seguras mediante 'threading.Lock'
    para garantizar la integridad del estado en entornos multi-hilo.
- opcua_io.py:
    Capa de abstracción de hardware. Gestiona la conexión con el servidor
    OPC UA del PLC, permitiendo lecturas/escrituras síncronas con una caché local.
- behaviour.py:
    Contiene la lógica operacional dividida en comportamientos cíclicos
    independientes (Behaviours).
- resource_agent.py:
    Clase principal del agente que orquestra la conexión XMPP, la inicialización
    del hardware y el despliegue de los comportamientos.
- main.py:
    Punto de entrada asíncrono que gestiona el ciclo de vida de la aplicación.

3. CAPAS DE LA ARQUITECTURA
---------------------------

A. CAPA DE COMUNICACIÓN (OPC UA):
   - Mapea variables físicas del PLC (sensores y actuadores) a nombres lógicos.
   - Provee funciones para forzar variables virtuales (Overrides) durante
     maniobras de compensación.

B. CAPA DE ESTADOS (STATE MANAGER):
   - Estados: INIT, IDLE, RUNNING, FAULT_DETECTED, COMPENSATING, STOPPED, EMERGENCY.
   - Regla Estricta: El sistema solo puede volver a RUNNING tras una falla si
     pasa primero por el estado de COMPENSATING.

C. CAPA DE COMPORTAMIENTOS (REASONING LAYER):
   1. MonitoringBehaviour: Actualiza la imagen de los datos del PLC cada 100ms.
   2. ControlBehaviour: Gestiona transiciones básicas (Start/Stop/Reset) y
      coordina con las señales de parada de emergencia físicas.
   3. FaultDetectionBehaviour (Soft-Sensor):
      Utiliza algoritmos de Dead Reckoning para monitorear la cinemática.
      Detecta fallas comparando señales de comando (Motor ON) vs retroalimentación
      temporal (Tiempo de tránsito de pieza).
   4. CompensationBehaviour:
      Ejecuta algoritmos de recuperación. Inyecta comandos de bypass al PLC
      cuando se detectan fallas eléctricas o lógicas menores.
   5. MLObserverBehaviour (Shadow Mode):
      Implementa un modelo de Machine Learning (Isolation Forest) que observa
      tiempos de ciclo de forma pasiva. Detecta anomalías predictivas por
      degradación mecánica sin interferir en el control directo.

4. FLUJO DE DETECCIÓN Y RECUPERACIÓN (CASO 2.1)
-----------------------------------------------
1. El sensor virtual detecta que la pieza no se mueve a pesar de que el PLC
   ordenó el arranque.
2. El estado cambia a FAULT_DETECTED.
3. El CompensationBehaviour toma el control, cambia el estado a COMPENSATING.
4. El agente activa el bit de 'Override' y fuerza el arranque manual ('bAgent_Cmd_Conveyor').
5. Una vez recuperada la pieza, limpia comandos, resetea alarmas en el PLC y
   devuelve el sistema a RUNNING.

5. INTEGRACIÓN DE INTELIGENCIA ARTIFICIAL
-----------------------------------------
- Algoritmo: Isolation Forest (scikit-learn).
- Entrenamiento: Fase inicial de 20 ciclos "sanos" (Auto-calibración).
- Inferencia: Evaluación en tiempo real de cada ciclo de transporte.
- Propósito: Mantenimiento Predictivo. Alerta sobre fricción o lentitud
  atípica antes de que el sensor virtual de seguridad dispare una falla real.

6. SEGURIDAD Y CONCURRENCIA
---------------------------
- Thread-Safety: Uso de Locks en el StateManager para evitar condiciones de carrera.
- Fail-Safe: Si la compensación falla o se detecta un atasco físico insalvable,
  el agente transiciona a STOPPED, exigiendo intervención humana.
================================================================================

# Behavior.py


from spade.behaviour import PeriodicBehaviour
from agent.states import AgentState
from sklearn.ensemble import IsolationForest
import numpy as np
import asyncio
import time

# ============================================================
# BASE BEHAVIOUR
# ============================================================
class BaseBehaviour(PeriodicBehaviour):
    def __init__(self, state_manager, io, period=0.5):
        super().__init__(period=period)
        self.state_manager = state_manager
        self.io = io

# ============================================================
# MONITORING BEHAVIOUR
# ============================================================
class MonitoringBehaviour(BaseBehaviour):
    async def run(self):
        # Mantiene la caché del OPC UA actualizada
        self.io.read_all()

# ============================================================
# HEARTBEAT BEHAVIOUR (Nuevo)
# ============================================================
class HeartbeatBehaviour(BaseBehaviour):
    def __init__(self, state_manager, io, period=0.2):
        super().__init__(state_manager, io, period=period)
        self.pulse_state = False

    async def run(self):
        # Solo envía heartbeat si estamos conectados/activos (opcional)
        self.pulse_state = not self.pulse_state
        self.io.write_bool("bAgent_Pulse", self.pulse_state)

# ============================================================
# CONTROL BEHAVIOUR
# ============================================================
class ControlBehaviour(BaseBehaviour):
    def __init__(self, state_manager, io, period=0.2):
        super().__init__(state_manager, io, period=period)
        self.last_enable_state = False

    async def run(self):
        data = self.io.read_all()
        state = self.state_manager.get_state()
        if not data: return

        # Variables adaptadas a la estación Storage
        estop_active = data.get("xStore_Error", False)        # Falla general / E-Stop
        reset_pressed = data.get("xStore_ClearFault", False)  # Botón de rearme
        current_enable = data.get("MES_ACTIVO", True)         # Asumimos True si no lo controlas desde MES

        # 1. Gestión de Emergencias
        if estop_active:
            if state != AgentState.EMERGENCY:
                print("🚨 [CONTROL] ERROR DE ESTACIÓN (xStore_Error ACTIVADO)")
                self.state_manager.set_state(AgentState.EMERGENCY)
            return

        if state == AgentState.EMERGENCY and reset_pressed:
            print("✅ [CONTROL] Sistema rearmado tras error.")
            self.last_enable_state = True
            self.state_manager.set_state(AgentState.IDLE)
            return

        # 2. Arranque inicial
        if state == AgentState.INIT:
            self.state_manager.set_state(AgentState.IDLE)
            self.last_enable_state = current_enable
            return

        # 3. Transiciones RUNNING / IDLE
        if state == AgentState.IDLE and current_enable and not self.last_enable_state:
            print("🚀 [CONTROL] Agente en RUNNING")
            self.state_manager.set_state(AgentState.RUNNING)

        if state == AgentState.RUNNING and not current_enable:
            print("⏸️ [CONTROL] Agente en IDLE")
            self.state_manager.set_state(AgentState.IDLE)
        
        self.last_enable_state = current_enable

 # ============================================================
# PREDICTIVE TRACKER BEHAVIOUR (Doble Timer Coordinado)
# ============================================================
class PredictiveTrackerBehaviour(BaseBehaviour):
    def __init__(self, state_manager, io, period=0.01): # Lectura cada 10ms
        super().__init__(state_manager, io, period=period)
        
        # ⏱️ CONFIGURACIÓN DE LOS DOS TIMERS
        # Timer 1: Flanco de Subida (El que ACTIVA la compensación)
        self.T_TRAVEL_RISING_MS = 1500  # <--- Ajusta este valor (suele ser mayor porque inicia antes)
        self.t_travel_rising_sec = self.T_TRAVEL_RISING_MS / 1000.0
        
        # Timer 2: Flanco de Bajada (Solo informativo/testigo)
        self.T_TRAVEL_FALLING_MS = 1050
        self.t_travel_falling_sec = self.T_TRAVEL_FALLING_MS / 1000.0
        
        # Variables de estado independiente para cada timer
        self.rising_timer_active = False
        self.transit_start_rising = 0.0
        
        self.falling_timer_active = False
        self.transit_start_falling = 0.0

        # Memoria general
        self.cable_fault_alerted = False
        self.blind_mode = False       
        
        # 🎯 MEMORIA DE FLANCO (Estado del ciclo anterior)
        self.last_x_start = False

    async def run(self):
        if self.state_manager.get_state() != AgentState.RUNNING:
            self.rising_timer_active = False
            self.falling_timer_active = False
            return

        data = self.io.read_all()
        if not data: return

        # Lectura de los sensores
        x_start = data.get("xConv_start", False)
        x_mid = data.get("xConv_mid", True)
        current_time = time.time()

        # ---------------------------------------------------------
        # 1. MONITOREO DE DAÑOS EN REPOSO
        # ---------------------------------------------------------
        if x_mid == False and not self.cable_fault_alerted and not (self.rising_timer_active or self.falling_timer_active):
            print("🚨 [SOFT-SENSOR] Sensor central tapado sin pieza en camino. Modo Ciego activado.")
            self.cable_fault_alerted = True
        elif x_mid == True and self.cable_fault_alerted:
            print("✅ [SOFT-SENSOR] Sensor central físico restablecido.")
            self.cable_fault_alerted = False

        # ---------------------------------------------------------
        # 2. DETECCIÓN DE FLANCOS Y ARRANQUE DE TIMERS
        # ---------------------------------------------------------
        
        # A. FLANCO DE SUBIDA (La pieza entra al sensor)
        if self.last_x_start == False and x_start == True:
            if not self.rising_timer_active:
                print(f"🟢 [SOFT-SENSOR] FLANCO DE SUBIDA: Cronómetro PRINCIPAL de {self.T_TRAVEL_RISING_MS}ms INICIADO.")
                self.rising_timer_active = True
                self.transit_start_rising = current_time
                self.blind_mode = self.cable_fault_alerted

        # B. FLANCO DE BAJADA (La pieza sale del sensor)
        if self.last_x_start == True and x_start == False:
            if not self.falling_timer_active:
                print(f"🔴 [SOFT-SENSOR] FLANCO DE BAJADA: Cronómetro SECUNDARIO de {self.T_TRAVEL_FALLING_MS}ms INICIADO.")
                self.falling_timer_active = True
                self.transit_start_falling = current_time

        # Guardamos el estado para el siguiente milisegundo
        self.last_x_start = x_start

        # ---------------------------------------------------------
        # 3. EJECUCIÓN CINEMÁTICA DE LOS TIMERS
        # ---------------------------------------------------------
        
        # --- VERIFICAR TIMER 1 (FLANCO DE SUBIDA -> COMPENSA) ---
        if self.rising_timer_active:
            elapsed_rising = current_time - self.transit_start_rising
            
            if elapsed_rising >= self.t_travel_rising_sec:
                # El tiempo de la cabeza de la pieza se cumplió
                if self.blind_mode == True or x_mid == True:
                    print(f"⚠️ [SOFT-SENSOR] Timer PRINCIPAL cumplido ({elapsed_rising*1000:.0f}ms). ¡INYECTANDO PULSO COMPENSATORIO!")
                    
                    # ⚡ Inyección asíncrona (Solo el flanco de subida tiene permiso de hacer esto)
                    self.io.write_bool("bAgent_Override_xConv_mid", True)
                    await asyncio.sleep(0.15)
                    self.io.write_bool("bAgent_Override_xConv_mid", False)
                    
                    print("⚡ [SOFT-SENSOR] OVERRIDE RETIRADO: Compensación exitosa.")
                else:
                    print(f"✅ [SOFT-SENSOR] Timer PRINCIPAL cumplido ({elapsed_rising*1000:.0f}ms). Hardware ok.")
                
                # Apagamos este timer
                self.rising_timer_active = False

        # --- VERIFICAR TIMER 2 (FLANCO DE BAJADA -> SOLO MONITOREA) ---
        if self.falling_timer_active:
            elapsed_falling = current_time - self.transit_start_falling
            
            if elapsed_falling >= self.t_travel_falling_sec:
                # El tiempo de la cola de la pieza se cumplió
                if self.blind_mode == True or x_mid == True:
                    print(f"ℹ️ [TESTIGO] Timer SECUNDARIO de bajada cumplido ({elapsed_falling*1000:.0f}ms). La cola de la pieza debería estar pasando ahora.")
                else:
                    print(f"ℹ️ [TESTIGO] Timer SECUNDARIO de bajada cumplido ({elapsed_falling*1000:.0f}ms).")
                
                # Apagamos este timer sin inyectar nada
                self.falling_timer_active = False

# ============================================================
# FAULT DETECTION BEHAVIOUR (Arranques fallidos / Atascos)
# ============================================================
class FaultDetectionBehaviour(BaseBehaviour):
    def __init__(self, state_manager, io, period=0.2):
        super().__init__(state_manager, io, period=period)
        self.conv_timer = None

    async def run(self):
        if self.state_manager.get_state() != AgentState.RUNNING:
            self.conv_timer = None
            return

        data = self.io.read_all()
        if not data: return

        x_start = data.get("xConv_start", False)
        motor_on = data.get("G1KF1_A1", False)

        if x_start:
            if self.conv_timer is None:
                self.conv_timer = time.time()
            
            elapsed = time.time() - self.conv_timer

            # CASO 2.2: Atasco físico grave
            if motor_on and elapsed > 4.0:
                print(f"⚠️ [FAULT] Detectado: Atasco en Cinta ({round(elapsed,1)}s)")
                self.agent.current_fault = "Fail_Conveyor_Jam"
                self.state_manager.set_state(AgentState.FAULT_DETECTED)
            
            # CASO 2.1: Falla eléctrica/PLC
            elif not motor_on and elapsed > 2.1:
                print(f"⚠️ [FAULT] Detectado: Falla de Arranque del Motor ({round(elapsed,1)}s)")
                self.agent.current_fault = "Fail_Conveyor_NotStarted"
                self.state_manager.set_state(AgentState.FAULT_DETECTED)
        else:
            self.conv_timer = None

# ============================================================
# COMPENSATION / RECOVERY BEHAVIOUR (Lógica Inteligente)
# ============================================================
class CompensationBehaviour(BaseBehaviour):
    async def run(self):
        if self.state_manager.get_state() != AgentState.FAULT_DETECTED:
            return

        fault = getattr(self.agent, "current_fault", None)
        if not fault: return

        print("⚙️ [STATE] Iniciando maniobra de compensación...")
        self.state_manager.set_state(AgentState.COMPENSATING)

        # 🛑 CASO 2.2: NO COMPENSABLE
        if fault == "Fail_Conveyor_Jam":
            print("\n" + "🟥"*20)
            print("🛑 👮 ALERTA CRÍTICA: ATASCO EN CINTA")
            print("🛑 Condición NO recuperable automáticamente.")
            print("🛑 Retirar pieza manualmente y presionar RESET.")
            print("🟥"*20 + "\n")
            
            self.state_manager.set_state(AgentState.STOPPED)
            return

        # 🔧 CASO 2.1: COMPENSABLE
        if fault == "Fail_Conveyor_NotStarted":
            print("\n🔧 [RECOVERY] Modo: Falla de Arranque (Inyectando Override)")
            
            self.io.write_bool("Override", True)
            await asyncio.sleep(0.5)

            print("   ⚙️ Forzando variable bAgent_Cmd_Conveyor -> TRUE")
            self.io.write_bool("bAgent_Cmd_Conveyor", True)
            
            await asyncio.sleep(2.5)

            print("   🧹 Limpiando comandos virtuales...")
            self.io.write_bool("bAgent_Cmd_Conveyor", False)
            self.io.write_bool("Override", False)
            
            print("   ⚡ Enviando pulso de Reset...")
            self.io.write_bool("xStore_ClearFault", True)
            await asyncio.sleep(0.5)
            self.io.write_bool("xStore_ClearFault", False)

            print("🏭 [AUTO-START] Maniobra exitosa. Retornando a RUNNING.")
            self.agent.current_fault = None
            self.state_manager.set_state(AgentState.RUNNING)

# ============================================================
# ML PREDICTIVE OBSERVER (Shadow Mode)
# ============================================================
class MLObserverBehaviour(BaseBehaviour):
    def __init__(self, state_manager, io, period=0.5):
        super().__init__(state_manager, io, period=period)
        self.cycle_start_time = None
        self.history_data = []
        
        self.model = IsolationForest(contamination=0.1, random_state=42)
        self.is_trained = False
        self.training_threshold = 20

    async def run(self):
        if self.state_manager.get_state() != AgentState.RUNNING:
            self.cycle_start_time = None
            return

        data = self.io.read_all()
        if not data: return

        x_start = data.get("xConv_start", False)
        motor_on = data.get("G1KF1_A1", False)

        if x_start and motor_on and self.cycle_start_time is None:
            self.cycle_start_time = time.time()

        elif not x_start and not motor_on and self.cycle_start_time is not None:
            cycle_duration = time.time() - self.cycle_start_time
            self.cycle_start_time = None

            if cycle_duration < 0.5:
                return

            if not self.is_trained:
                self.history_data.append([cycle_duration])
                print(f"📊 [ML SHADOW] Observando ciclo {len(self.history_data)}/{self.training_threshold} (Duración: {round(cycle_duration, 2)}s)")

                if len(self.history_data) >= self.training_threshold:
                    print("🧠 [ML SHADOW] Entrenando modelo Isolation Forest con datos históricos...")
                    X_train = np.array(self.history_data)
                    self.model.fit(X_train)
                    self.is_trained = True
                    print("✅ [ML SHADOW] Modelo entrenado. Iniciando monitoreo predictivo.")
            else:
                X_test = np.array([[cycle_duration]])
                prediction = self.model.predict(X_test)

                if prediction[0] == -1:
                    print(f"🔮 ⚠️ [MANTENIMIENTO PREDICTIVO] Anomalía detectada.")
                    print(f"   ↳ El ciclo duró {round(cycle_duration, 2)}s. Posible degradación mecánica o fricción inusual.")

# OPCUA.py

import time
from opcua import Client, ua

class OPCUAIO:
    # ⚠️ AHORA APUNTA AL PLC REAL (.12) EN LUGAR DE LA HMI (.13)
    def __init__(self, url="opc.tcp://192.168.1.13:4840"):
        self.url = url
        self.client = Client(url)
        self.nodes = {}
        self.namespace = 4
        self._cache = {}

        # ---------------------------------------------------------
        # 🕵️ LISTA DE PREFIJOS A PROBAR (PLC FESTO CODESYS V3)
        # ---------------------------------------------------------
        self.POSSIBLE_PREFIXES = [
            "|var|CPX-CEC-C1-V3.Application.GVL_Storage.",
            "|var|CPX-CEC-C1-V3.Application.PLC_PRG." # Añadido como respaldo común
        ]

        # ---------------------------------------------------------
        # 📋 LISTA DE VARIABLES (Estación Storage + IA Soft-Sensors)
        # ---------------------------------------------------------
        self.var_names = [
            # --- CONTROL GENERAL ---
            "MES_ACTIVO",           
            "MES_AUTO",      
            "MES_Start",    
            "MODO",            
            "PASO_INIT",            
            
            # --- FALLOS Y ESTADOS ---
            "xStore_Error",                
            "xStore_Busy",       
            "xStore_Full",     
            "AL_MES_STOP",    
            
            # --- SENSORES (Inputs) ---
            "xWPAvailable",
            "xWPmetallic",
            "xWPnotBlack",
            "xConv_start",
            "xConv_mid",       # Ajustado a minúscula para coincidir con el agente
            "xConv_End",

            # --- ACTUADORES FÍSICOS E INDICADORES ---
            "indLightGn",        
            "indLightRd",
            "indLightYe",
            "G1KF1_A1",        # NUEVO: Estado del Motor del Conveyor (Para ML y Atascos)

            # --- MÉTRICAS Y COMANDOS ESTÁNDAR ---
            "AL_MES_OEE",    
            "AL_MES_Q",  
            "AL_MES_AV",
            "AL_MES_EFI",    
            "xStore_ClearFault",
            
            # --- GEMELO DIGITAL & SOFT-SENSORS (Agente IA) ---
            "bAgent_Pulse",               # Out: Heartbeat
            "bAgent_Override_xConv_mid",  # Out: Sensor virtual inyectado para Dead Reckoning
            "bPhysical_Sensor_Fault_xConv_mid"  # 🟢 NUEVA: Señal de falla explícita del PLC

        ]

    def connect(self):
        try:
            print(f"⌛ [OPCUA] Conectando a {self.url} ...")
            self.client.connect()
            print("✅ [OPCUA] Conexión TCP establecida.")
            
            print(f"🔗 [OPCUA] Buscando variables en namespace {self.namespace}...")
            found_count = 0
            
            for name in self.var_names:
                var_found = False
                
                # BUCLE MÁGICO: Prueba prefijos hasta encontrar la variable
                for prefix in self.POSSIBLE_PREFIXES:
                    node_id = f"ns={self.namespace};s={prefix}{name}"
                    try:
                        node = self.client.get_node(node_id)
                        # Intentamos leer para validar existencia
                        node.get_value()
                        
                        # Si llegamos aquí, ¡LA ENCONTRAMOS!
                        self.nodes[name] = node
                        
                        short_prefix = "GVL" if "GVL" in prefix else "PLC_PRG"
                        print(f"   ✅ OK ({short_prefix}): {name}")
                        
                        var_found = True
                        found_count += 1
                        break
                    except:
                        continue

                if not var_found:
                    # Lo ponemos como Warning para no ensuciar tanto la consola
                    print(f"   ⚠️ NO ENCONTRADA: {name} (Revisa Symbol Configuration en CODESYS)")

            print(f"🟢 [OPCUA] Sistema listo. {found_count} variables operativas de {len(self.var_names)}.")

            if found_count == 0:
                raise Exception("CRÍTICO: No se encontró ninguna variable mapeada.")

        except Exception as e:
            print(f"🔴 [OPCUA] Error de conexión: {e}")
            raise e

    def read_all(self):
        """Lee todas las variables mapeadas y actualiza la caché"""
        try:
            for name, node in self.nodes.items():
                val = node.get_value()
                self._cache[name] = val
            return self._cache
        except Exception as e:
            print(f"⚠️ [OPCUA] Error leyendo: {e}")
            return {}

    def get_cached_data(self):
        return self._cache

    def write_bool(self, var_name, value):
        if var_name not in self.nodes:
            return False

        try:
            node = self.nodes[var_name]
            ua_val = ua.DataValue(ua.Variant(value, ua.VariantType.Boolean))
            node.set_value(ua_val)
            
            # 👇 MODIFICACIÓN: Silenciamos específicamente el Heartbeat (bAgent_Pulse)
            if value is True and var_name != "bAgent_Pulse":
                print(f"📝 [OPCUA] Escribiendo {value} en {var_name}")
                
            return True
        except Exception as e:
            print(f"❌ [OPCUA] Error escribiendo en {var_name}: {e}")
            return False

    def disconnect(self):
        try:
            self.client.disconnect()
            print("🔌 [OPCUA] Desconectado.")
        except:
            pass

# Resource_agent.py

import asyncio
from spade.agent import Agent

# Importamos nuestras herramientas
from agent.opcua_io import OPCUAIO
from agent.states import StateManager, AgentState
from agent.behaviour import (
    MonitoringBehaviour,
    HeartbeatBehaviour,           # 🟢 NUEVO: Latido para el PLC
    ControlBehaviour,
    PredictiveTrackerBehaviour,   # 🟢 NUEVO: Soft-Sensor y Dead Reckoning
    FaultDetectionBehaviour,
    CompensationBehaviour,
    MLObserverBehaviour
)

class ResourceAgent(Agent):
    def __init__(self, jid, password):
        super().__init__(jid, password)
        # Inicializamos el hardware virtual y la máquina de estados
        self.io = OPCUAIO()
        self.state_manager = StateManager()
        
        # Variable compartida para que el FaultDetection le avise al Compensation
        self.current_fault = None

    async def setup(self):
        print("\n🏭 [INGENIERÍA] Inicializando ResourceAgent (Storage + Soft-Sensors)...")
        try:
            # 1. Conexión física al PLC
            self.io.connect()
            
            # 2. Limpieza de variables virtuales por seguridad (anti-falsos arranques)
            self._clean_virtual_commands()
            
            # 3. Inyección de comportamientos (Sistema Nervioso)
            
            # Monitoreo general (100ms)
            self.add_behaviour(MonitoringBehaviour(self.state_manager, self.io, period=0.1))
            
            # Latido para mantener vivo el PLC (200ms)
            self.add_behaviour(HeartbeatBehaviour(self.state_manager, self.io, period=0.2))
            
            # Soft-Sensor Predictivo: Velocidad ultra rápida para no perder flancos (10ms)
            self.add_behaviour(PredictiveTrackerBehaviour(self.state_manager, self.io, period=0.01))
            
            # Control y Detección evalúan cada 200ms
            self.add_behaviour(ControlBehaviour(self.state_manager, self.io, period=0.2))
            self.add_behaviour(FaultDetectionBehaviour(self.state_manager, self.io, period=0.2))
            
            # Compensación evalúa cada 500ms (no necesita tanta prisa si no hay fallos)
            self.add_behaviour(CompensationBehaviour(self.state_manager, self.io, period=0.5))

            # Observador predictivo con ML cada 200ms
            self.add_behaviour(MLObserverBehaviour(self.state_manager, self.io, period=0.2))

            print("✅ [SISTEMA] Agente ensamblado y comportamientos en ejecución.")
            
            # 4. Transición inicial
            # Pasamos a IDLE para que el ControlBehaviour tome el mando y lo pase a RUNNING
            self.state_manager.set_state(AgentState.IDLE)
            
        except Exception as e:
            print(f"❌ [CRÍTICO] Error al iniciar el agente: {e}")
            await self.stop()

    def _clean_virtual_commands(self):
        print("🧹 [INIT] Asegurando variables de Override en OFF...")
        cmds = [
            "Override",
            "bAgent_Cmd_Conveyor",
            "xStore_ClearFault",
            "bAgent_Override_xConv_mid"  # 🟢 Añadido para limpiar el sensor virtual en el arranque
        ]
        for c in cmds:
            try:
                self.io.write_bool(c, False)
            except:
                pass

# Spade.py

from agent.io_interface import IOInterface

class SpadeIO(IOInterface):

    def __init__(self, agent):
        self.agent = agent

    def read(self, name):
        # leer desde mensajes XMPP
        pass

    def write(self, name, value):
        # enviar comando vía SPADE
        pass


# State_manager.py

from agent.states import AgentState
from threading import Lock

class StateManager:
    def __init__(self):
        # El agente siempre inicia en el estado de inicialización
        self._state = AgentState.INIT
        self._lock = Lock()
        
        # =====================================================================
        # 🚦 DICCIONARIO DE TRANSICIONES ESTRICTAS
        # =====================================================================
        # Lo mantenemos como diccionario de clase para poder modificarlo
        # dinámicamente con add_transition() si en el futuro se requiere.
        self._allowed_transitions = {
            AgentState.INIT: [
                AgentState.IDLE,
                AgentState.EMERGENCY  # Necesario si arranca con la seta física pulsada
            ],

            AgentState.IDLE: [
                AgentState.RUNNING,
                AgentState.SHUTDOWN,
                AgentState.EMERGENCY
            ],

            AgentState.RUNNING: [
                AgentState.FAULT_DETECTED,
                AgentState.STOPPED,
                AgentState.IDLE,      # Para paradas normales/directas
                AgentState.EMERGENCY
            ],

            AgentState.FAULT_DETECTED: [
                AgentState.COMPENSATING, # <-- Paso crítico para el Gemelo Digital
                AgentState.STOPPED,
                AgentState.EMERGENCY
            ],

            AgentState.COMPENSATING: [
                AgentState.RUNNING,      # <-- Retorno a producción tras recuperar
                AgentState.STOPPED,
                AgentState.EMERGENCY
            ],

            AgentState.STOPPED: [
                AgentState.IDLE,
                AgentState.SHUTDOWN,
                AgentState.EMERGENCY
            ],

            AgentState.EMERGENCY: [
                AgentState.IDLE,      # CRÍTICO: Permite salir de emergencia tras el reset
                AgentState.SHUTDOWN
            ],

            AgentState.SHUTDOWN: []
        }

    def get_state(self) -> AgentState:
        """Devuelve el estado actual de forma segura."""
        return self._state

    def add_transition(self, source: AgentState, dest: AgentState):
        """
        Permite registrar nuevas transiciones dinámicamente desde el Agente.
        Soluciona el error 'object has no attribute add_transition'.
        """
        with self._lock:
            if source not in self._allowed_transitions:
                self._allowed_transitions[source] = []
            
            if dest not in self._allowed_transitions[source]:
                self._allowed_transitions[source].append(dest)
                # print(f"[STATE_MGR] Regla dinámica añadida: {source.name} -> {dest.name}")

    def set_state(self, new_state: AgentState) -> bool:
        """
        Intenta cambiar el estado. Retorna True si la transición es válida y exitosa.
        """
        with self._lock:
            if self._is_valid_transition(self._state, new_state):
                old_state = self._state
                self._state = new_state
                print(f"🚦 [STATE] {old_state.name} → {new_state.name}")
                return True
            else:
                # Si falla, imprimimos el error en consola pero NO cambiamos el estado
                print(
                    f"❌ [STATE ERROR] Transición inválida rechazada por seguridad: "
                    f"{self._state.name} → {new_state.name}"
                )
                return False

    def _is_valid_transition(self, current: AgentState, new: AgentState) -> bool:
        """Verifica internamente si el salto está permitido en el diccionario."""
        return new in self._allowed_transitions.get(current, [])

# States.py

from enum import Enum
from threading import Lock

class AgentState(Enum):
    INIT = "INIT"
    IDLE = "IDLE"
    RUNNING = "RUNNING"
    FAULT_DETECTED = "FAULT_DETECTED"
    COMPENSATING = "COMPENSATING"
    STOPPED = "STOPPED"          # <-- Agregado para el caso de atasco no compensable
    EMERGENCY = "EMERGENCY"
    SHUTDOWN = "SHUTDOWN"        # <-- Agregado por si apagas el agente

class StateManager:
    def __init__(self):
        self._current_state = AgentState.INIT
        self._lock = Lock()  # 🔒 Protege el estado cuando varios comportamientos escriben a la vez

        # Definimos las transiciones permitidas (Origen, Destino) en un Set
        self._allowed_transitions = {
            # Desde INIT
            (AgentState.INIT, AgentState.IDLE),
            (AgentState.INIT, AgentState.EMERGENCY),
            
            # Desde IDLE
            (AgentState.IDLE, AgentState.RUNNING),
            (AgentState.IDLE, AgentState.EMERGENCY),
            (AgentState.IDLE, AgentState.SHUTDOWN),
            
            # Desde RUNNING
            (AgentState.RUNNING, AgentState.IDLE),
            (AgentState.RUNNING, AgentState.FAULT_DETECTED),
            (AgentState.RUNNING, AgentState.EMERGENCY),
            (AgentState.RUNNING, AgentState.STOPPED),
            
            # Desde FAULT_DETECTED
            (AgentState.FAULT_DETECTED, AgentState.COMPENSATING),
            (AgentState.FAULT_DETECTED, AgentState.EMERGENCY),
            (AgentState.FAULT_DETECTED, AgentState.STOPPED),
            
            # Desde COMPENSATING
            (AgentState.COMPENSATING, AgentState.RUNNING),      # Éxito de la reparación
            (AgentState.COMPENSATING, AgentState.EMERGENCY),    # Emergencia durante arreglo
            (AgentState.COMPENSATING, AgentState.IDLE),         # Rendición / Cancelación
            (AgentState.COMPENSATING, AgentState.STOPPED),      # Falla crítica (pasa a detenido)
            
            # Desde STOPPED
            (AgentState.STOPPED, AgentState.IDLE),              # Operador rearmó la máquina
            (AgentState.STOPPED, AgentState.EMERGENCY),
            (AgentState.STOPPED, AgentState.SHUTDOWN),

            # Desde EMERGENCY
            (AgentState.EMERGENCY, AgentState.IDLE),            # Recuperación a espera
            (AgentState.EMERGENCY, AgentState.RUNNING),         # Recuperación directa
            (AgentState.EMERGENCY, AgentState.SHUTDOWN),
        }

    def get_state(self):
        # Lectura segura con Lock
        with self._lock:
            return self._current_state

    def set_state(self, new_state):
        # Escritura segura con Lock
        with self._lock:
            if new_state == self._current_state:
                return True
                
            if (self._current_state, new_state) in self._allowed_transitions:
                print(f"🚦 [STATE] {self._current_state.name} → {new_state.name}")
                self._current_state = new_state
                return True
            else:
                print(f"❌ [STATE ERROR] Transición inválida rechazada: {self._current_state.name} → {new_state.name}")
                return False

    def add_transition(self, source, dest):
        """Permite inyectar transiciones extra dinámicamente si el código lo requiere"""
        with self._lock:
            self._allowed_transitions.add((source, dest))